# 05 · Análise, tabela final e frase para o manuscrito

**Sessão P1 · CPU** (Colab sem GPU, ou localmente). Lê `resultados/ablacao_cabeca` e `resultados/baselines`, confere a integridade e aplica a **regra de decisão pré-registrada**:

* métrica primária: AUC (de probabilidades), média entre as 5 dobras;
* comparação primária: KAN (d) × a **melhor** MLP (b ou c), pareada por dobra;
* "a KAN melhora" = ΔAUC > 0 em ≥ 4/5 dobras **e** IC 95 % do ΔAUC OOF (bootstrap de casos) excluindo 0; com uma semente só, a frase ganha a ressalva "single training seed".

Saídas em `resultados/`: `tabela_final.csv`, `tabela_final.tex`, `decisao.json`, `frase_manuscrito.md`, `roc_cabecas.png`.

In [ ]:
# ==== Configuração ====
NO_COLAB = True    # False para rodar localmente apontando para uma cópia dos resultados
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"
DIR_RESULTADOS = f"{DIR_DRIVE}/resultados"   # localmente: caminho da pasta resultados/
SHA_PROTOCOLO_ESPERADO = "b7cd9baca53edd96c855cca10914fc258efa8ddfd0937cc0d3e9572de5ef9e13"
N_BOOT = 2000

In [ ]:
if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, roc_curve
if NO_COLAB:
    sys.path.insert(0, DIR_DRIVE)
import protocolo_comum as pc
R = DIR_RESULTADOS
E1, E2 = pc.CONFIGS_EXP1, pc.CONFIGS_EXP2
NOMES = {E1["a"]: "ResNet152 + Linear (a)", E1["b"]: "ResNet152 + MLP-128 (b)", E1["c"]: "ResNet152 + MLP-iso (c)",
         E1["d"]: "ResNet152 + KAN (d) = TransKAN", E1["e"]: "ResNet152 + KAN-8 (e)"}
NOMES_BB = {"alexnet": "AlexNet", "mobilenet_v2": "MobileNetV2", "mnasnet1_3": "MNASNet1.3",
            "shufflenet_v2_x0_5": "ShuffleNetV2 x0.5", "squeezenet1_0": "SqueezeNet1.0", "densenet169": "DenseNet169",
            "maxvit_t": "MaxViT-T", "swin_b": "Swin-B", "efficientnet_v2_l": "EfficientNetV2-L", "vgg16": "VGG16",
            "regnet_y_16gf": "RegNetY-16GF"}
for b, n in NOMES_BB.items():
    NOMES[E2[b]] = n
runs = pd.concat([pc.ler_resultados(R, e) for e in ("ablacao_cabeca", "baselines")], ignore_index=True)
runs = runs[~runs["teste_rapido"].astype(bool)].copy()
print(len(runs), "unidades concluídas")
display(runs.groupby(["experimento", "config", "semente"]).size().rename("dobras"))

## 1. Integridade

In [ ]:
problemas, avisos = [], []
for col in ("protocolo_sha256", "dobras_sha256", "precisao"):
    vals = runs[col].dropna().unique()
    if len(vals) != 1:
        problemas.append(f"{col} não é único: {vals}")
if runs["protocolo_sha256"].iloc[0] != SHA_PROTOCOLO_ESPERADO:
    problemas.append("SHA do protocolo diferente do publicado")
runs["reserva"] = ~runs["gpu"].fillna("").str.contains("L4")
if runs["reserva"].any():
    print("† unidades fora da L4:", runs.loc[runs.reserva, ["config", "semente", "dobra", "gpu"]].to_dict("records"))
# recalcular as métricas a partir das predições e conferir com o JSON
oof = []
for _, r in runs.iterrows():
    p = pd.read_csv(Path(r["caminho"]) / "pred_teste.csv")
    y, s = p["y_es"].to_numpy(), p["p_es_1"].to_numpy()
    pred = (p[["p_es_0", "p_es_1"]].to_numpy().argmax(1))
    rec = {"es_acc": accuracy_score(y, pred), "es_auc": roc_auc_score(y, s), "es_rec_macro": recall_score(y, pred, average="macro"),
           "es_prec_macro": precision_score(y, pred, average="macro", zero_division=0), "es_f1_macro": f1_score(y, pred, average="macro")}
    for k, v in rec.items():
        if abs(v - r[k]) > 1e-9:
            problemas.append(f"{r['config']} s{r['semente']} d{r['dobra']}: {k} recalculado {v} ≠ {r[k]}")
    # O bug antigo (AUC de classes) já seria pego acima, pois a AUC é recalculada das probabilidades.
    # AUC = revocação macro aqui só por coincidência, p. ex. num modelo que colapsou numa classe.
    if abs(r["es_auc"] - r["es_rec_macro"]) < 1e-12:
        avisos.append(f"{r['config']} s{r['semente']} d{r['dobra']}: AUC = revocação macro = {r['es_auc']:.3f} "
                      f"(modelo degenerado? {len(np.unique(np.round(s, 6)))} valores distintos de P(saída))")
    oof.append(p.assign(config=r["config"], semente=r["semente"], dobra=r["dobra"]))
oof = pd.concat(oof, ignore_index=True)
print("Problemas:", problemas or "nenhum")
print("Avisos:", avisos or "nenhum")
assert not problemas

## 2. Média ± dp entre dobras (cada dobra = média das sementes disponíveis)

In [ ]:
METR = ["es_acc", "es_prec_macro", "es_rec_macro", "es_f1_macro", "es_auc"]
por_dobra = runs.groupby(["config", "dobra"])[METR + ["treino_min"]].mean().reset_index()
sementes = runs.groupby("config")["semente"].apply(lambda s: sorted(s.unique())).rename("sementes")
agg = por_dobra.groupby("config")[METR + ["treino_min"]].agg(["mean", "std", "count"])
agg.columns = [f"{a}_{b}" for a, b in agg.columns]
dp_sementes = (runs.groupby(["config", "semente"])["es_auc"].mean().groupby("config").std(ddof=1)).rename("dp_auc_entre_sementes")
info = runs.groupby("config")[["params_cabeca", "params_total", "reserva"]].agg({"params_cabeca": "first", "params_total": "first", "reserva": "any"})
resumo = agg.join(sementes).join(dp_sementes).join(info)
display(resumo.round(4))

## 3. KAN × MLPs: diferença pareada por dobra e bootstrap de casos (OOF)

In [ ]:
def oof_medio(cfg):
    """P(saída) por imagem, média das sementes disponíveis (OOF: cada imagem em uma dobra)."""
    o = oof[oof.config == cfg].groupby(["rel_path", "case_id", "y_es", "dobra"])["p_es_1"].mean().reset_index()
    return o.set_index("rel_path")

def comparar(cfg_kan, cfg_mlp, n_boot=N_BOOT, semente=0):
    a, b = oof_medio(cfg_kan), oof_medio(cfg_mlp)
    comuns = a.index.intersection(b.index)
    a, b = a.loc[comuns], b.loc[comuns]
    y = a["y_es"].to_numpy()
    d_oof = roc_auc_score(y, a.p_es_1) - roc_auc_score(y, b.p_es_1)
    casos = a["case_id"].to_numpy()
    uc = np.unique(casos)
    idx_caso = {c: np.flatnonzero(casos == c) for c in uc}
    rng = np.random.default_rng(semente)
    boots = []
    for _ in range(n_boot):
        amostra = np.concatenate([idx_caso[c] for c in rng.choice(uc, size=len(uc), replace=True)])
        yy = y[amostra]
        if yy.min() == yy.max():
            continue
        boots.append(roc_auc_score(yy, a.p_es_1.to_numpy()[amostra]) - roc_auc_score(yy, b.p_es_1.to_numpy()[amostra]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    pa = por_dobra[por_dobra.config == cfg_kan].set_index("dobra")["es_auc"]
    pb = por_dobra[por_dobra.config == cfg_mlp].set_index("dobra")["es_auc"]
    dd = (pa - pb).dropna()
    return {"kan": cfg_kan, "mlp": cfg_mlp, "delta_auc_oof": float(d_oof), "ic95": [float(lo), float(hi)],
            "delta_auc_dobras": dd.round(4).to_dict(), "media_delta_dobras": float(dd.mean()),
            "dp_delta_dobras": float(dd.std(ddof=1)) if len(dd) > 1 else None,
            "dobras_kan_vence": int((dd > 0).sum()), "n_dobras": int(len(dd)),
            "sementes_kan": sementes.get(cfg_kan), "sementes_mlp": sementes.get(cfg_mlp)}

comps = {}
for k in ("b", "c", "a"):
    if E1["d"] in resumo.index and E1[k] in resumo.index:
        comps[k] = comparar(E1["d"], E1[k])
        print(k, json.dumps(comps[k], default=str))

## 4. Regra de decisão pré-registrada e frase

In [ ]:
def fmt(cfg, m="es_auc"):
    r = resumo.loc[cfg]
    return f"{r[m + '_mean']:.3f}±{r[m + '_std']:.3f}"

mlps = [k for k in ("b", "c") if k in comps]
assert mlps, "Faltam as MLPs (b, c) para decidir."
melhor = max(mlps, key=lambda k: resumo.loc[E1[k], "es_auc_mean"])
c = comps[melhor]
uma_semente = len(c["sementes_kan"] or []) < 2 or len(c["sementes_mlp"] or []) < 2
kan_melhora = c["dobras_kan_vence"] >= 4 and c["ic95"][0] > 0
kan_pior = c["dobras_kan_vence"] <= 1 and c["ic95"][1] < 0
desfecho = "A" if kan_melhora else ("C" if kan_pior else "B")
X, Y = fmt(E1[melhor]), fmt(E1["d"])
d, (lo, hi), k = c["delta_auc_oof"], c["ic95"], c["dobras_kan_vence"]
ressalva = " (single training seed)" if uma_semente else ""
FRASES = {
    "A": (f"Under an identical protocol (same ResNet-152 backbone, folds, augmentation and two-phase schedule), the KAN head "
          f"raised AUC from {X} to {Y} relative to the best MLP head (paired ΔAUC = {d:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}]; "
          f"better in {k}/5 folds){ressalva}, including an MLP with a matched 3.9 M-parameter budget. The improvement is "
          f"therefore attributable to the head, although it is small compared with the effect of ImageNet pre-training."),
    "B": (f"Under an identical protocol{ressalva}, the KAN head performed on par with linear and MLP heads "
          f"(AUC {Y} vs. {X}; ΔAUC = {d:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}] includes zero). TransKAN's advantage over "
          f"the baselines of Lira et al. is explained by the pre-trained ResNet-152 backbone and the two-phase fine-tuning "
          f"recipe rather than by the KAN head itself; we present the KAN head as a competitive alternative, not as the source of the gain."),
    "C": (f"Under an identical protocol{ressalva}, the KAN head underperformed the MLP heads "
          f"(AUC {Y} vs. {X}; ΔAUC = {d:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}]; better in {k}/5 folds). TransKAN's advantage over "
          f"the baselines of Lira et al. is explained by the pre-trained ResNet-152 backbone and the two-phase fine-tuning "
          f"recipe rather than by the KAN head itself."),
}
inv_e2 = {v: k for k, v in E2.items()}
bb_feitos = sorted({inv_e2[c] for c in runs.loc[runs.experimento == "baselines", "config"] if c in inv_e2})
transkan_auc = resumo.loc[E1["d"], "es_auc_mean"]
vence_todos = all(transkan_auc > resumo.loc[E2[b], "es_auc_mean"] for b in bb_feitos)
proibicoes = {
    "so_trocar_a_cabeca_melhora": desfecho == "A",
    "new_SOTA": vence_todos and len(bb_feitos) == len(pc.BASELINES),
    "outperforms_transformer_baselines": {"swin_b", "maxvit_t"} <= set(bb_feitos)
        and all(transkan_auc > resumo.loc[E2[b], "es_auc_mean"] for b in ("swin_b", "maxvit_t")),
    "parameter_efficiency": False,
}
decisao = {"desfecho": desfecho, "melhor_mlp": E1[melhor], "uma_semente": uma_semente, "comparacoes": comps,
           "frase": FRASES[desfecho], "pode_afirmar": proibicoes, "baselines_reavaliados": bb_feitos,
           "baselines_nao_reavaliados": [b for b in pc.BASELINES if b not in bb_feitos]}
pc.gravar_json_atomico(decisao, Path(R) / "decisao.json")
print("Desfecho:", desfecho, "| melhor MLP:", NOMES[E1[melhor]])
print("\n" + FRASES[desfecho])
print("\nPode afirmar:", proibicoes)

## 5. Tabela final (CSV + LaTeX)

In [ ]:
def celula(cfg, m):
    r = resumo.loc[cfg]
    sd = r[m + '_std']
    return f"{r[m + '_mean']:.3f} ± {sd:.3f}" if pd.notna(sd) else f"{r[m + '_mean']:.3f}"

linhas = []
painel_a = [E1[k] for k in "abcde" if E1[k] in resumo.index]
painel_b = [E2[b] for b in pc.BASELINES if E2[b] in resumo.index]
for painel, cfgs in (("A", painel_a), ("B", [E1["a"]] + painel_b + [E1["d"]])):
    for cfg in cfgs:
        if cfg not in resumo.index:
            continue
        r = resumo.loc[cfg]
        nome = NOMES.get(cfg, cfg) + (" †" if r["reserva"] else "")
        if painel == "B" and cfg == E1["a"]:
            nome = "ResNet152" + (" †" if r["reserva"] else "")
        linhas.append({"painel": painel, "modelo": nome, "config": cfg,
                       "params_cabeca_M": r["params_cabeca"] / 1e6, "params_total_M": r["params_total"] / 1e6,
                       "acc": celula(cfg, "es_acc"), "prec_macro": celula(cfg, "es_prec_macro"),
                       "rec_macro": celula(cfg, "es_rec_macro"), "f1_macro": celula(cfg, "es_f1_macro"),
                       "auc": celula(cfg, "es_auc"), "treino_min_dobra": round(r["treino_min_mean"], 1),
                       "sementes": ",".join(map(str, r["sementes"])), "n_dobras": int(r["es_auc_count"])})
tabela = pd.DataFrame(linhas)
tabela.to_csv(Path(R) / "tabela_final.csv", index=False)
display(tabela)

def tex_pm(s):
    return "$" + s.replace(" ± ", r" \pm ") + "$"
tex = [r"\begin{table*}[t]", r"\centering",
       r"\caption{Entry/exit classification under a single protocol (grouped stratified 5-fold by case, same two-phase "
       r"recipe, last epoch). Mean $\pm$ s.d. across folds; AUC from predicted probabilities. Micro-averaged P/R/F1 equal "
       r"accuracy and macro specificity equals macro recall in the binary case, so they are omitted. Time: training minutes "
       r"per fold on an NVIDIA L4" + (r"; $\dagger$ = run on another GPU" if tabela.modelo.str.contains("†").any() else "") + r".}",
       r"\label{tab:protocolo_unico}", r"\footnotesize",
       r"\begin{tabular}{lrcccccr}", r"\hline",
       r"Model & Params (M) & Acc & Prec. & Rec. & F1 & AUC & Min \\", r"\hline"]
for painel, titulo in (("A", "Head ablation (ResNet-152 backbone)"), ("B", "Backbones (linear head)")):
    tex.append(r"\multicolumn{8}{l}{\textit{" + titulo + r"}} \\")
    for _, r in tabela[tabela.painel == painel].iterrows():
        nome = r["modelo"].replace("†", r"$^\dagger$").replace("_", r"\_")
        tex.append(f"{nome} & {r['params_total_M']:.2f} & {tex_pm(r['acc'])} & {tex_pm(r['prec_macro'])} & "
                   f"{tex_pm(r['rec_macro'])} & {tex_pm(r['f1_macro'])} & {tex_pm(r['auc'])} & {r['treino_min_dobra']:.0f} \\\\")
    tex.append(r"\hline")
tex += [r"\end{tabular}", r"\end{table*}"]
(Path(R) / "tabela_final.tex").write_text("\n".join(tex) + "\n", encoding="utf-8")
print("\n".join(tex))

## 6. Frase, ROC e diagnóstico da KAN

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for k in "abcd":
    if E1[k] in resumo.index:
        o = oof_medio(E1[k])
        fpr, tpr, _ = roc_curve(o["y_es"], o["p_es_1"])
        ax.plot(fpr, tpr, label=f"{NOMES[E1[k]]} (AUC OOF {roc_auc_score(o['y_es'], o['p_es_1']):.3f})")
ax.plot([0, 1], [0, 1], "k:", lw=0.8)
ax.set_xlabel("1 − especificidade"); ax.set_ylabel("sensibilidade (saída)"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(Path(R) / "roc_cabecas.png", dpi=200)
diag = runs.filter(like="kan_").join(runs[["config"]]).groupby("config").mean().dropna(how="all")
print("Fração das entradas da KAN dentro da grade [−1, 1] (teste):")
display(diag.round(3))
texto = f"""# Frase para o manuscrito (desfecho {decisao['desfecho']})

{decisao['frase']}

- Comparação primária: KAN (d) × {NOMES[decisao['melhor_mlp']]}; ΔAUC OOF = {c['delta_auc_oof']:+.3f}, IC 95 % [{c['ic95'][0]:+.3f}, {c['ic95'][1]:+.3f}]; KAN vence em {c['dobras_kan_vence']}/{c['n_dobras']} dobras.
- Pode afirmar: {json.dumps(decisao['pode_afirmar'])}
- Baselines não reavaliados (saem da tabela ou ficam marcados): {decisao['baselines_nao_reavaliados'] or 'nenhum'}
- Correções obrigatórias no texto: substituir os números antigos da TransKAN; remover SMOTE+ENN; checkpoint = última época; dobras por caso estratificadas por strat4 com semente fixa; baselines pré-treinados no mesmo protocolo; AUC por probabilidade; "two-layer KAN"; contagens de imagens e casos do notebook 00; citar Lira et al. (2025, IJLM) como origem da base (GuWID = FDCPUnBGunshotDB) e da tabela de referência.
"""
(Path(R) / "frase_manuscrito.md").write_text(texto, encoding="utf-8")
print(texto)